In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

In [ ]:
Definitions for RDM Service were reviewed and corrected following discussion with Mallie and Evan, mainly to resolve confusion around BU ID vs BU Name. Monday board was updated accordingly and the SharePoint ingestion mapping is now ready.

Completed in Fabric dataflow:
service_id, service_src_id, service_src_name, service_src_sys_inst_id, service_name_conformed, service_bu_id, service_mstr_service_id, z_src_is_active, z_src_created_date_time, z_src_created_by_user, z_src_modified_date_time, z_src_modified_by_user.

Remaining to be populated in UDM:
z_src_sys_inst_id, z_record_created_by_user, z_record_created_date_time, z_record_modified_by_user, z_record_modified_date_time, z_record_is_active.

entity ------> care_Epi_contract_id

In [ ]:
Task completed in DEV.
Created and configured a new Dataflow Gen2 for RDM - Business Unit using the existing SharePoint RDM template.

Implemented SharePoint to Silver ingestion for the Type 1 RDM table and created target table:
silver_rdm_business_unit in vhgfbc01dev_silver lakehouse.

Columns created / mapped in the dataflow:

bu_id from SharePoint ID
bu_name_conformed
bu_name_short_conformed
z_src_is_active
z_src_created_date_time from SharePoint Created
z_src_created_by_user from SharePoint Created By.title
z_src_modified_date_time from SharePoint Modified
z_src_modified_by_user from SharePoint Modified By.title

Columns ignored / not created in the dataflow:

Version
z_record_created_date_time
z_record_created_by_user
z_record_modified_date_time
z_record_modified_by_user
z_record_is_active

Reason:
z_record_* fields were not created in the dataflow because these are platform / UDM-generated audit fields, not source SharePoint fields. This task only required SharePoint source columns to be ingested into Silver as part of a Type 1 RDM implementation. No _add table, Power Automate, or notebook changes were required.

In [ ]:
Hi Team,

I’m in the process of applying for an Italian Schengen visa and require an employment reference letter as part of the application.

Could you please provide a letter on company letterhead addressed as below:

To,
The Visa Officer
Consulate General of Italy
United Kingdom

The letter should include:

- Company name, address, and contact details
- My full name (as per passport)
- My current designation/role
- My employment start date

Please let me know if you need any additional information from my side, and I would appreciate it if this can be provided at the earliest.

Thanks in advance for your help.

Best regards,
[Your Full Name]

MPB-------------

In [ ]:
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'MPB001'
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(ten.id AS varchar(100))

In [ ]:
Updated MPB care_epi_contr_id mapping in the Care Episode notebook to use the new RDM Contracts table. The old source-derived contract key logic was replaced with a join to silver_rdm_contract using contr_src_sys_inst_id = 'MPB001' and contr_src_id = ten.id, and care_epi_contr_id is now populated from rdmc.contr_id as per the updated Monday definition.

In [ ]:
wip ---------------------------

In [ ]:
DevOps wording

Updated WIP care_epi_contr_id mapping to use the new RDM Contracts table. Replaced the old source-derived customer-name logic with a join to silver_rdm_contract using contr_src_sys_inst_id = 'WIP001' and contr_src_name = ahrd.customer_name, and now populate care_epi_contr_id from rdmc.contr_id.

Validation wording

Validated WIP contract mapping in select-based testing. Matching rows successfully returned rdmc.contr_id as care_epi_contr_id using the new RDM Contracts join.

Retained the existing silver_contract join as it is still used for other contract-derived attributes, while introducing silver_rdm_contract specifically for the new care_epi_contr_id mapping.

In [ ]:
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
-- Join WIP contract to the new RDM Contracts table using source system instance and source contract name
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'WIP001'
   AND trim(lower(rdmc.contr_src_name)) = trim(lower(ahrd.customer_name))

In [ ]:
SELECT
    care_epi_id,
    care_epi_src_id,
    care_epi_contr_id,
    z_src_system_instance
FROM
    silver_care_episode
WHERE
    z_src_system_instance = 'WIP001'
    AND care_epi_contr_id IS NOT NULL
LIMIT 100;

In [ ]:
SELECT
    COUNT(*) AS total_count,
    SUM(CASE WHEN care_epi_contr_id IS NOT NULL THEN 1 ELSE 0 END) AS populated_count,
    SUM(CASE WHEN care_epi_contr_id IS NULL THEN 1 ELSE 0 END) AS null_count,
    COUNT(DISTINCT care_epi_contr_id) AS distinct_contract_id_count
FROM silver_care_episode
WHERE z_src_system_instance = 'WIP001';

In [ ]:
string(outputs('Run_a_query_against_a_dataset')?['body']?['error']?['pbi.error']?['details']?[0]?['detail']?['value'])

In [ ]:
-- Empty table create karaychi query
CREATE TABLE shape_on_list_test_add (
    test_source_name STRING,
    test_source_id INT
);

In [ ]:
INSERT INTO shape_on_list_test_add (test_source_name, test_source_id)
VALUES
    ('Pathway A', 1),
    ('Pathway B', 2),
    ('Pathway C', 3),
    ('Pathway D', 4);

In [ ]:
sn----------------------

In [ ]:
SELECT
    r.id AS care_epi_src_id,
    r.id_organisation_source,
    CONCAT('SONE', r.id_organisation_source) AS expected_src_sys_inst_id,
    CAST(r.id_organisation_source AS varchar(100)) AS expected_contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_src_id,
    rdmc.contr_id AS care_epi_contr_id
FROM
    silver_sone_srreferralin r
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))
WHERE
    r.id_organisation_source IS NOT NULL
LIMIT 100;

In [ ]:
SELECT
    r.id AS care_epi_src_id,
    r.id_organisation_source,
    CONCAT('SONE', r.id_organisation_source) AS expected_src_sys_inst_id,
    CAST(r.id_organisation_source AS varchar(100)) AS expected_contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_src_id,
    rdmc.contr_id AS care_epi_contr_id
FROM
    silver_sone_srreferralin r
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))
WHERE
    r.id_organisation_source IS NOT NULL
LIMIT 100;

In [ ]:
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
-- Join SONE contract to the new RDM Contracts table using dynamic source system instance and source contract ID
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))

In [ ]:
SELECT
    COUNT(*) AS total_count,
    SUM(CASE WHEN care_epi_contr_id IS NOT NULL THEN 1 ELSE 0 END) AS populated_count,
    SUM(CASE WHEN care_epi_contr_id IS NULL THEN 1 ELSE 0 END) AS null_count
FROM silver_care_episode
WHERE z_src_system_id = 'SONE';

In [ ]:
if(
  equals(outputs('Compose_Failstep'),'Run_a_query_against_a_dataset'),
  string(outputs('Run_a_query_against_a_dataset')?['body']?['error']?['pbi.error']?['details']?[0]?['detail']?['value']),
  if(
    equals(outputs('Compose_Failstep'),'Get_items'),
    string(outputs('Get_items')?['body']),
    if(
      equals(outputs('Compose_Failstep'),'Create_item'),
      string(outputs('Create_item')?['body']),
      'Unable to capture exact error message'
    )
  )
)

In [ ]:
neww------------

In [ ]:
SELECT
    r.id AS care_epi_src_id,
    r.id_organisation_source,
    CONCAT('SONE', r.id_organisation_source) AS expected_src_sys_inst_id,
    CAST(r.id_organisation_source AS varchar(100)) AS expected_contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_src_id,
    rdmc.contr_src_name,
    rdmc.contr_id AS care_epi_contr_id
FROM
    silver_sone_srreferralin r
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))
WHERE
    r.id_organisation_source IS NOT NULL
LIMIT 100;

In [ ]:
SELECT
    r.id AS care_epi_src_id,
    r.id_organisation_source,
    CONCAT('SONE', r.id_organisation_source) AS expected_src_sys_inst_id,
    CAST(r.id_organisation_source AS varchar(100)) AS expected_contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_src_id,
    rdmc.contr_src_name,
    rdmc.contr_id AS care_epi_contr_id
FROM
    silver_sone_srreferralin r
INNER JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))
WHERE
    r.id_organisation_source IS NOT NULL
LIMIT 100;

In [ ]:
last

In [ ]:
SELECT
    r.id AS care_epi_src_id,
    r.id_organisation_source,
    CONCAT('SONE', r.id_organisation_source) AS expected_src_sys_inst_id,
    CAST(r.id_organisation_source AS varchar(100)) AS expected_contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_src_id,
    rdmc.contr_src_name,
    rdmc.contr_id AS care_epi_contr_id
FROM
    silver_sone_srreferralin r
INNER JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
   AND CAST(rdmc.contr_src_id AS varchar(100)) = CAST(r.id_organisation_source AS varchar(100))
WHERE
    r.id_organisation_source IS NOT NULL
LIMIT 100;

In [ ]:
SELECT
    COUNT(*) AS total_count,
    SUM(CASE WHEN care_epi_contr_id IS NOT NULL THEN 1 ELSE 0 END) AS populated_count,
    SUM(CASE WHEN care_epi_contr_id IS NULL THEN 1 ELSE 0 END) AS null_count
FROM silver_care_episode
WHERE z_src_system_id = 'SONE';

In [ ]:
SELECT
    id_organisation_source,
    care_epi_contr_id,
    COUNT(*) AS row_count
FROM silver_care_episode
WHERE z_src_system_id = 'SONE'
GROUP BY id_organisation_source, care_epi_contr_id
ORDER BY id_organisation_source, row_count DESC;

In [ ]:
SELECT
    COUNT(DISTINCT care_epi_contr_id) AS distinct_contract_id_count
FROM silver_care_episode
WHERE z_src_system_id = 'SONE';

In [ ]:
SELECT DISTINCT
    id_organisation_source,
    CONCAT('SONE', id_organisation_source) AS expected_src_sys_inst_id,
    care_epi_contr_id
FROM silver_care_episode
WHERE z_src_system_id = 'SONE'
ORDER BY id_organisation_source;

In [ ]:
Updated SONE care_epi_contr_id mapping to use the new RDM Contracts table. Replaced the old source-derived organisation-based value with a join to silver_rdm_contract using dynamic source system instance (CONCAT('SONE', r.id_organisation_source)) and source contract ID (r.id_organisation_source), and now populate care_epi_contr_id from rdmc.contr_id. Validation confirmed the join returned matching contract IDs, and post-run counts show 415469 total SONE records, 415469 populated, and 0 nulls.